In [1]:
# JsonOutputParser
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

model = ChatOpenAI( 
    # model="unsloth/gemma-4-e2b-it",
    model="qwen3-30b-a3b-instruct-2507",
    base_url="http://...:.../v1",
    api_key="lm-studio",
)

In [2]:
# 원하는 데이터 구조 정의
class FinancialAdvice(BaseModel):
    setup: str = Field(description="금융 조언 상황을 설정하기 위한 질문")
    advice: str = Field(description="질문을 해결하기 위한 금융 답변")

In [3]:
# JSON 출력 파서 설정 및 프롬프트 템플릿에 지침 삽입
parser = JsonOutputParser(pydantic_object=FinancialAdvice)
prompt = PromptTemplate(
    template="다음 금융 관련 질문에 답변해주세요. \n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

In [4]:
# 체인 구성: 프롬프트 -> 모델 -> 파서
chain = prompt | model | parser

# 체인 실행
chain.invoke({"query": "부동산에 관련하여 금융 조언을 받을 수 있게 질문하라."})

{'setup': '부동산 투자에 대해 금융 조언이 필요합니다. 현재 주택 구입을 고려하고 있으며, 대출 금리와 월 상환금액, 그리고 장기적인 자산 가치 증가 가능성에 대해 알고 싶습니다.',
 'advice': '부동산 투자 시 대출 금리는 현재 시장 조건과 신용등급에 따라 결정되며, 월 상환금액은 대출 금액, 이자율, 대출 기간을 고려하여 계산됩니다. 장기적으로는 지역 개발과 수요 증가로 인해 부동산 가치가 오를 가능성이 있으나, 시장 변동성과 과잉 공급 위험도 고려해야 합니다. 또한, 자산의 유지 관리 비용과 세금 부담을 포함한 총 소유 비용을 미리 계산하는 것이 중요합니다.'}